In [ ]:
!pip -q install -U transformers accelerate datasets peft bitsandbytes

In [ ]:


import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_DIR = "/content/drive/MyDrive/ai_nlp3/umi_sft_lora/final"  # ✅ 저장 경로

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

print("✅ base model loaded")

from peft import PeftModel

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

print("✅ LoRA adapter loaded:", ADAPTER_DIR)

def make_diary_from_messages(messages, fish_level=3, max_new_tokens=400):
    system = (
        "너는 텍스트 기반 잠입수사 게임 Project: UMI_PROTOCOL의 스토리 에이전트다. "
        "입력은 하루의 대화 로그(messages)이며, 이를 바탕으로 '일기'만 작성한다. "
        "톤은 어둡고 불안하며 잠입수사 기록처럼 건조해야 한다. "
        "밝은/훈훈/희망적 표현 금지. "
        "fish_level이 높을수록 감각 왜곡(어안렌즈, 비린내, 청각 왜곡 등)을 더 반영한다. "
        "출력은 JSON이 아니라 '일기 본문 텍스트만' 출력한다."
    )

    user = (
        f"[fish_level={fish_level}]\n"
        "아래 messages 로그만 근거로 일기를 작성해. 새 사실 창작 금지.\n"
        "조건: 7~10문장, 줄바꿈 없이 한 덩어리로.\n\n"
        f"{messages}"
    )

    chat = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return gen


# ✅ 예시 테스트 로그 (너 규칙 반영)
test_messages = [
    {"msg_id":"m1","speaker":"NPC_전광어","content":"오늘 밤엔 B4 쪽을 비우지 마라. 기도보다 중요한 일이 있다."},
    {"msg_id":"m2","speaker":"PLAYER","content":"무슨 일이지?"},
    {"msg_id":"m3","speaker":"NPC_청갈치","content":"괜히 묻지 마. 여기선 질문이 흔적이 돼."},
    {"msg_id":"m4","speaker":"PLAYER","content":"(녹음기를 켠다)"},
    {"msg_id":"m5","speaker":"NPC_이민어","content":"너도 들었지? 금속 긁는 소리. 벽 안쪽에서."},
    {"msg_id":"m6","speaker":"PLAYER","content":"어느 벽?"},
    {"msg_id":"m7","speaker":"NPC_박복어","content":"[뻐끔] 그 문… 열지 마…"},
    {"msg_id":"m8","speaker":"PLAYER","content":"(B4로 이동한다)"},
    {"msg_id":"m9","speaker":"NPC_곽빙어","content":"열쇠는 하나뿐이야. 그리고 누군가 이미 복사했어."},
]

diary = make_diary_from_messages(test_messages, fish_level=3)
print(diary)

##아래는 옵션 3개 후보 출력
for i in range(3):
    print(f"\n--- sample {i+1} ---")
    print(make_diary_from_messages(test_messages, fish_level=4, max_new_tokens=450))
